# 02 — Temporal EDA

**Owner**: Deekshitha (C5)  
**Phase**: 1 — Data Ingestion & EDA

This notebook explores temporal patterns in the Pittsburgh EMS & Fire dispatch data.

**Topics covered:**
- Quarterly call volume trends
- Year-over-year comparison
- Seasonal patterns (quarter-level heatmap)
- Priority distribution over time
- Service type trends over time

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 20)
print('✅ Imports ready')

## 1. Load Data

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PARENT_DIR = PROJECT_ROOT.parent

ems_csv = DATA_DIR / 'EMS_Data.csv'
ems_xlsx = PARENT_DIR / 'EMS_Data.xlsx'

if ems_csv.exists():
    df_ems = pd.read_csv(ems_csv)
    df_fire = pd.read_csv(DATA_DIR / 'Fire_Data.csv')
elif ems_xlsx.exists():
    print('Loading from Excel...')
    df_ems = pd.read_excel(ems_xlsx)
    df_fire = pd.read_excel(PARENT_DIR / 'Fire_Data.xlsx')
else:
    raise FileNotFoundError('No data files found.')

df = pd.concat([df_ems, df_fire], ignore_index=True)
print(f'Total records: {len(df):,}')

## 2. Quarterly Call Volume Trends

In [ ]:
# Create a sortable period column
quarter_map = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}
df['quarter_num'] = df['call_quarter'].map(quarter_map)
df['period'] = df['call_year'].astype(str) + '-' + df['call_quarter']
df['period_sort'] = df['call_year'] * 10 + df['quarter_num']

quarterly = df.groupby(['period', 'period_sort']).size().reset_index(name='count')
quarterly = quarterly.sort_values('period_sort')

fig = px.line(
    quarterly,
    x='period',
    y='count',
    title='Quarterly Call Volume — All Services',
    labels={'period': 'Year-Quarter', 'count': 'Number of Calls'},
    markers=True,
)
fig.update_layout(template='plotly_dark', width=1000, height=450)
fig.show()

## 3. Year-over-Year Comparison by Service Type

In [ ]:
quarterly_svc = df.groupby(['period', 'period_sort', 'service']).size().reset_index(name='count')
quarterly_svc = quarterly_svc.sort_values('period_sort')

fig = px.line(
    quarterly_svc,
    x='period',
    y='count',
    color='service',
    title='Quarterly Trends — EMS vs Fire',
    labels={'period': 'Year-Quarter', 'count': 'Call Count', 'service': 'Service'},
    color_discrete_map={'EMS': '#636EFA', 'Fire': '#EF553B'},
    markers=True,
)
fig.update_layout(template='plotly_dark', width=1000, height=450)
fig.show()

## 4. Seasonal Patterns — Quarter-Level Heatmap

In [ ]:
heatmap_data = df.groupby(['call_year', 'call_quarter']).size().reset_index(name='count')
heatmap_pivot = heatmap_data.pivot(index='call_year', columns='call_quarter', values='count')
# Reorder columns
heatmap_pivot = heatmap_pivot.reindex(columns=['Q1', 'Q2', 'Q3', 'Q4'])

fig = px.imshow(
    heatmap_pivot,
    title='Call Volume Heatmap — Year × Quarter',
    labels=dict(x='Quarter', y='Year', color='Call Count'),
    color_continuous_scale='YlOrRd',
    aspect='auto',
)
fig.update_layout(template='plotly_dark', width=700, height=500)
fig.show()

## 5. Priority Distribution Over Time

In [ ]:
# Top 5 priorities over time
top_priorities = df['priority'].value_counts().head(5).index.tolist()
df_top_pri = df[df['priority'].isin(top_priorities)]

pri_quarterly = df_top_pri.groupby(['period', 'period_sort', 'priority']).size().reset_index(name='count')
pri_quarterly = pri_quarterly.sort_values('period_sort')

fig = px.area(
    pri_quarterly,
    x='period',
    y='count',
    color='priority',
    title='Top 5 Priorities — Quarterly Stacked Area',
    labels={'period': 'Year-Quarter', 'count': 'Call Count', 'priority': 'Priority'},
)
fig.update_layout(template='plotly_dark', width=1000, height=500)
fig.show()

## 6. Year-over-Year Growth Rate

In [ ]:
yearly_counts = df.groupby('call_year').size()
yoy_growth = yearly_counts.pct_change() * 100

growth_df = pd.DataFrame({
    'year': yearly_counts.index,
    'total_calls': yearly_counts.values,
    'yoy_growth_pct': yoy_growth.values
})

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Bar(x=growth_df['year'], y=growth_df['total_calls'], name='Total Calls', marker_color='#636EFA'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=growth_df['year'], y=growth_df['yoy_growth_pct'], name='YoY Growth %',
               mode='lines+markers', line=dict(color='#FF6692', width=3)),
    secondary_y=True
)
fig.update_layout(title='Annual Volume & Year-over-Year Growth', template='plotly_dark', width=900, height=450)
fig.update_yaxes(title_text='Call Count', secondary_y=False)
fig.update_yaxes(title_text='YoY Growth (%)', secondary_y=True)
fig.show()

In [ ]:
print('=== Temporal Summary ===')
print(f'Year range: {df["call_year"].min()} — {df["call_year"].max()}')
print(f'Total quarters of data: {df["period"].nunique()}')
print(f'Avg quarterly volume: {quarterly["count"].mean():,.0f}')
print(f'Max quarterly volume: {quarterly["count"].max():,} ({quarterly.loc[quarterly["count"].idxmax(), "period"]})')
print(f'Min quarterly volume: {quarterly["count"].min():,} ({quarterly.loc[quarterly["count"].idxmin(), "period"]})')